In [0]:
%python
import hashlib
from datetime import datetime, UTC
from pyspark.sql import functions as F

def stable_long_id(*parts):
    raw_value = "||".join(str(part) for part in parts)
    return int(hashlib.sha256(raw_value.encode("utf-8")).hexdigest()[:15], 16)


def add_audit_columns(df, source_schema, table_name, run_id):
    source_object = f"{source_schema}.{table_name}" if source_schema else table_name
    return (
        df.withColumn("_ingest_run_id", F.lit(run_id))
        .withColumn("_ingest_ts", F.current_timestamp())
        .withColumn("_ingest_date", F.to_date(F.col("_ingest_ts")))
        .withColumn("_source_object", F.lit(source_object))
    )


def build_task_audit_details(
    table_name,
    run_id,
    layer,
    target_table,
    load_type="OVERWRITE",
    row_count=None,
    status="IN_PROGRESS",
    source_schema=None,
    start_ts=None,
    end_ts=None,
    message=None,
    watermark_from=None,
    watermark_to=None,
    rows_inserted=None,
    rows_updated="0",
    rows_deleted="0",
):
    timestamp_now = datetime.now(UTC)
    effective_start_ts = start_ts or timestamp_now
    effective_end_ts = end_ts
    effective_message = message or f"{source_schema + '.' if source_schema else ''}{table_name} -> {target_table}"
    effective_rows_read = None if row_count is None else str(row_count)
    effective_rows_inserted = effective_rows_read if rows_inserted is None else str(rows_inserted)

    return {
        "task_run_id": stable_long_id(run_id, table_name, layer),
        "run_id": stable_long_id(run_id),
        "entity_id": table_name,
        "layer": layer,
        "load_type": load_type,
        "watermark_from": watermark_from,
        "watermark_to": watermark_to,
        "rows_read": effective_rows_read,
        "rows_inserted": effective_rows_inserted,
        "rows_updated": str(rows_updated) if rows_updated is not None else None,
        "rows_deleted": str(rows_deleted) if rows_deleted is not None else None,
        "status": status,
        "start_ts": effective_start_ts,
        "end_ts": effective_end_ts,
        "message": effective_message,
    }


def write_task_audit(spark, audit_details, audit_task_table):
    audit_task_schema = spark.table(audit_task_table).schema
    spark.createDataFrame([audit_details], schema=audit_task_schema).write.format("delta").mode("append").saveAsTable(audit_task_table)
